In [ ]:
# Install required packages (run once)
!pip install -q ultralytics pillow flask flask-cors pyngrok opencv-python-headless
# Note: ultralytics will pull torch if not present; Colab GPU runtime typically already has torch.

In [ ]:
# Try downloading a public YOLOv8 weights file (example). If you have your own weights, upload them via Colab Files or Drive.
!wget -q -O yolov8l-seg.pt "https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8l-seg.pt" || true

# If you want much lower latency, use a smaller model like yolov8n.pt:
# !wget -q -O yolov8n.pt "https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt" || true

In [ ]:
# Load ngrok credentials from Colab Secrets or the runtime environment.
# Keep the auth token out of the notebook.
import os
from pyngrok import ngrok

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN") if userdata else None
except Exception:
    NGROK_AUTH_TOKEN = None

try:
    NGROK_DOMAIN = userdata.get("NGROK_DOMAIN") if userdata else None
except Exception:
    NGROK_DOMAIN = None

NGROK_AUTH_TOKEN = (NGROK_AUTH_TOKEN or os.environ.get("NGROK_AUTH_TOKEN", "")).strip()
NGROK_DOMAIN = (NGROK_DOMAIN or os.environ.get("NGROK_DOMAIN", "") or "bridgid-unstarting-niels.ngrok-free.dev").strip()

if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "Create a Colab Secret named NGROK_AUTH_TOKEN before starting ngrok."
    )

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print(f"ngrok token configured. Reserved domain: {NGROK_DOMAIN or 'none'}")

In [ ]:
# Flask server that loads YOLO and exposes a live frame-analysis endpoint
import base64
import io
import os
import socket
import time
from collections import Counter

from flask import Flask, jsonify, render_template_string, request
from flask_cors import CORS
from PIL import Image, ImageDraw, ImageFont

import numpy as np
import torch
from ultralytics import YOLO

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 8 * 1024 * 1024
CORS(app)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Keep the higher-accuracy reference model by default. Override this in the
# runtime environment when benchmarking a smaller real-time model.
YOLO_WEIGHTS = os.environ.get("YOLO_WEIGHTS", "yolov8l-seg.pt")
print("Loading YOLO model:", YOLO_WEIGHTS)
yolo = YOLO(YOLO_WEIGHTS)

NGROK_DOMAIN = (globals().get("NGROK_DOMAIN") or "").strip()
NGROK_DOMAIN = (
    NGROK_DOMAIN
    .removeprefix("https://")
    .removeprefix("http://")
    .rstrip("/")
)


def find_available_port(preferred_port=5000):
    """Use port 5000 when possible, otherwise choose an available port."""
    preferred_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        preferred_socket.bind(("0.0.0.0", preferred_port))
        return preferred_port
    except OSError:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as free_socket:
            free_socket.bind(("0.0.0.0", 0))
            return int(free_socket.getsockname()[1])
    finally:
        preferred_socket.close()


def class_name_for(class_id):
    names = yolo.names
    if isinstance(names, dict):
        return str(names.get(class_id, class_id))
    return str(names[class_id]) if 0 <= class_id < len(names) else str(class_id)


def pluralize(name):
    irregular = {
        "person": "people",
        "mouse": "mice",
        "sheep": "sheep",
    }
    if name in irregular:
        return irregular[name]
    if name.endswith(("s", "x", "z", "ch", "sh")):
        return f"{name}es"
    return f"{name}s"


def with_article(name):
    return f"an {name}" if name[:1].lower() in "aeiou" else f"a {name}"


def build_caption(detections):
    counts = Counter(item["class"] for item in detections)
    if not counts:
        return "I could not detect a clear object. Hold steady and try again."

    parts = []
    for name, count in counts.most_common(6):
        parts.append(with_article(name) if count == 1 else f"{count} {pluralize(name)}")

    if len(parts) == 1:
        objects = parts[0]
    elif len(parts) == 2:
        objects = f"{parts[0]} and {parts[1]}"
    else:
        objects = f"{', '.join(parts[:-1])}, and {parts[-1]}"

    return f"I can see {objects}."


def detect_and_annotate(pil_img, conf_thresh=0.45, annotate=False):
    image_array = np.array(pil_img)
    results = yolo(image_array, conf=conf_thresh, verbose=False)

    detections = []
    annotated = pil_img.copy() if annotate else None
    draw = ImageDraw.Draw(annotated) if annotated is not None else None

    font = None
    if annotate:
        try:
            font = ImageFont.truetype("DejaVuSans.ttf", 14)
        except Exception:
            font = ImageFont.load_default()

    for result in results:
        boxes = getattr(result, "boxes", None)
        if boxes is None:
            continue

        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
            confidence = float(box.conf[0].cpu().numpy())
            class_id = int(box.cls[0].cpu().numpy())
            class_name = class_name_for(class_id)

            if confidence < conf_thresh:
                continue

            detections.append({
                "class": class_name,
                "confidence": confidence,
                "bbox": [
                    float(x1),
                    float(y1),
                    float(x2 - x1),
                    float(y2 - y1),
                ],
            })

            if draw is not None:
                draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
                label = f"{class_name}: {confidence:.2f}"
                try:
                    draw.text(
                        (x1, max(0, y1 - 16)),
                        label,
                        fill="white",
                        font=font,
                    )
                except Exception:
                    pass

    detections.sort(key=lambda item: item["confidence"], reverse=True)
    return detections, annotated


@app.route("/")
def index():
    return render_template_string(
        "<h3>YOLO API is running.</h3>"
        "<p>POST frames to /analyze_frame</p>"
    )


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "device": device,
        "yolo_model": YOLO_WEIGHTS,
    })


@app.route("/analyze_frame", methods=["POST"])
def analyze_frame():
    frame_file = request.files.get("frame")

    if frame_file is None:
        return jsonify({
            "error": "No frame uploaded. Expected multipart field named 'frame'."
        }), 400

    try:
        image_bytes = frame_file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception as error:
        return jsonify({
            "error": f"Invalid image: {error}"
        }), 400

    try:
        confidence_value = float(request.form.get("confidence", "0.45"))
    except ValueError:
        confidence_value = 0.45

    confidence_value = min(max(confidence_value, 0.05), 0.95)
    started_at = time.perf_counter()
    include_annotated = request.args.get("include_annotated") == "1"

    detections, annotated = detect_and_annotate(
        image,
        conf_thresh=confidence_value,
        annotate=include_annotated,
    )

    response = {
        "caption": build_caption(detections),
        "detections": detections,
        "processing_time_ms": round(
            (time.perf_counter() - started_at) * 1000,
            1,
        ),
        "timestamp": int(time.time() * 1000),
    }

    if annotated is not None:
        buffer = io.BytesIO()
        annotated.save(buffer, format="JPEG", quality=70)
        response["annotated_image"] = base64.b64encode(
            buffer.getvalue()
        ).decode("utf-8")

    return jsonify(response)


def start_ngrok_and_run():
    from pyngrok import ngrok

    port = find_available_port(5000)

    if port != 5000:
        print(f"Port 5000 is already in use. Using port {port} instead.")

    ngrok.kill()

    try:
        tunnel_options = {
            "bind_tls": True,
        }

        if NGROK_DOMAIN:
            tunnel_options["domain"] = NGROK_DOMAIN

        tunnel = ngrok.connect(
            addr=port,
            proto="http",
            **tunnel_options,
        )

    except Exception as error:
        if not NGROK_DOMAIN:
            raise

        print(
            f"Could not bind reserved domain '{NGROK_DOMAIN}': {error}"
        )
        print("Starting a generated ngrok domain instead.")

        tunnel = ngrok.connect(
            addr=port,
            proto="http",
            bind_tls=True,
        )

    public_url = tunnel.public_url
    analysis_url = f"{public_url.rstrip('/')}/analyze_frame"

    print()
    print("=" * 70)
    print("ngrok public URL:")
    print(public_url)
    print()
    print("HTTP analysis endpoint:")
    print(analysis_url)
    print()
    print("Set this in the app environment:")
    print(f"EXPO_PUBLIC_SCENE_ANALYSIS_HTTP_URL={analysis_url}")
    print()
    print(f"Backend is ready on local port {port}.")
    print("Keep this cell running while using the Android app.")
    print("YOLO model:", YOLO_WEIGHTS)
    print("=" * 70)
    print()

    app.run(
        host="0.0.0.0",
        port=port,
        debug=False,
        use_reloader=False,
        threaded=True,
    )


start_ngrok_and_run()